# Basics &mdash; Powerset

**Concept 3 of the Basics decomposition:** *Powerset*

${\cal P}(S)$ is the set of all subsets; ${\cal P}(\emptyset)=\{\emptyset\}$, not $\emptyset$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Basics/Concept-Powerset/Concept-Powerset.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


$${\cal P}(S) = \{\,T \ :\ T \subseteq S\,\}$$

the set of **all** subsets, including $\emptyset$ and $S$ itself. Its size is
$|{\cal P}(S)| = 2^{|S|}$.

The trap: $${\cal P}(\emptyset) = \{\emptyset\}$$ which has **one** member, not
zero. $\emptyset$ is a subset of every set, including itself.

Powersets are everywhere in this book. **Subset construction** (Chapter 7) makes DFA
states out of ${\cal P}(Q)$, which is why determinizing an $n$-state NFA can cost
$2^n$ states. And $\mathcal{P}$ in a transition-function signature is precisely the
mark of **nondeterminism** (Concept 12).

## 2. Definitions

### Jove's `powset`

In [ ]:
print("powset({1,2,3}) :")
for t in sorted(powset({1, 2, 3}), key=lambda s: (len(s), sorted(s))):
    print("   ", set(t) if t else "{}")

### A hand-rolled version, to see the bit-vector correspondence

In [ ]:
def mypowset(S):
    xs = sorted(S)
    out = []
    for mask in range(2 ** len(xs)):
        out.append({xs[i] for i in range(len(xs)) if mask >> i & 1})
    return out

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;2.&nbsp;Set Builder Notation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Basics/Concept-Set-Builder-Notation/Concept-Set-Builder-Notation.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;4.&nbsp;Complement, Relative to a Universe](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Basics/Concept-Complement-Relative-To-Universe/Concept-Complement-Relative-To-Universe.ipynb)&nbsp;&rarr;

---

## 3. Tests

$|{\cal P}(S)| = 2^{|S|}$.

In [ ]:
for n in range(6):
    S = set(range(n))
    print("  |S| = %d -> |P(S)| = %2d   (2^%d = %d)"
          % (n, len(powset(S)), n, 2 ** n))
    assert len(powset(S)) == 2 ** n

**The trap:** ${\cal P}(\emptyset)$ has one member.

In [ ]:
p0 = powset(set())
print("P({}) =", p0, "   |P({})| =", len(p0))
assert len(p0) == 1
print("\nIts single member is the empty set.  P(empty) is NOT empty.")
print("And |P(empty)| = 2^0 = 1, exactly as the formula says.")

Each subset corresponds to a **bit vector** &mdash; which is where $2^n$ comes from.

In [ ]:
S = {'a', 'b', 'c'}
xs = sorted(S)
for mask in range(8):
    sub = {xs[i] for i in range(3) if mask >> i & 1}
    print("  %s -> %s" % (format(mask, '03b'), sorted(sub) if sub else '{}'))
mine = mypowset(S)
jove = [set(t) for t in powset(S)]
assert sorted(map(sorted, mine)) == sorted(map(sorted, jove))
print("\nhand-rolled agrees with Jove's powset")

Every member really is a subset, and $S$ and $\emptyset$ are both there.

In [ ]:
S = {1, 2, 3, 4}
P = [set(t) for t in powset(S)]
assert all(t <= S for t in P)
assert set() in P and S in P
print("all %d members are subsets; both extremes present" % len(P))

Why it matters: subset construction, and the $2^n$ blow-up of Chapter 7.

In [ ]:
for n in range(1, 8):
    print("  NFA with %d states -> at most %3d subset-construction DFA states"
          % (n, 2 ** n))
print("\nAnd P(Q) in a signature is the mark of nondeterminism:")
print("   DFA  delta : Q x Sigma -> Q")
print("   NFA  delta : Q x Sigma_eps -> P(Q)      <-- the P is the whole difference")

## 4. Exercises


1. What is ${\cal P}({\cal P}(\emptyset))$? How many members?
2. Is $\emptyset \in {\cal P}(S)$? Is $\emptyset \subseteq {\cal P}(S)$? Both?
3. How many subsets of a 10-element set have exactly 3 members?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Powerset')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')